In [ ]:
import pygame
import win32gui
import win32con
import socket
import time
import subprocess

# ==========================
# USER SETTINGS
# ==========================
DEFAULT_TRIM = -0.30
MAX_TRIM = 155

ROVER_SSID = "Rover"
ROVER_IP = "192.168.4.1"
ROVER_PORT = 4210

# ==========================
# UDP SOCKET
# ==========================
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.setblocking(False)

def send_udp(speedA, dirA, speedB, dirB):
    msg = f"{speedA},{dirA},{speedB},{dirB}"
    try:
        sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))
    except:
        pass

def send_heading(deg, duration_ms):
    msg = f"HEAD,{deg},{duration_ms}"
    try:
        sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))
    except:
        pass

def stop_all():
    send_udp(0, 0, 0, 0)

# ==========================
# WIFI CONTROL (WINDOWS)
# ==========================
def wifi_connect(ssid):
    try:
        subprocess.run(["netsh", "wlan", "connect", f"name={ssid}"], check=False)
    except:
        pass

def wifi_disconnect():
    try:
        subprocess.run(["netsh", "wlan", "disconnect"], check=False)
    except:
        pass

# ==========================
# Pygame Setup
# ==========================
pygame.init()
pygame.font.init()
clock = pygame.time.Clock()

background = pygame.image.load("nes.png")
image_width, image_height = background.get_size()

screen = pygame.display.set_mode((image_width, image_height), pygame.RESIZABLE)
pygame.display.set_caption("NES Controller Window")

hwnd = pygame.display.get_wm_info()["window"]
pygame.display.flip()
pygame.time.wait(100)

left, top, right, bottom = win32gui.GetWindowRect(hwnd)
window_width = right - left
window_height = bottom - top

display_info = pygame.display.Info()
screen_width = display_info.current_w

x_pos = screen_width - window_width
y_pos = 0

win32gui.SetWindowPos(
    hwnd,
    win32con.HWND_TOPMOST,
    x_pos,
    y_pos,
    window_width,
    window_height,
    win32con.SWP_SHOWWINDOW
)

# ==========================
# Text Printer
# ==========================
class TextPrint:
    def __init__(self):
        self.reset()
        self.font = pygame.font.Font(None, 25)

    def tprint(self, screen, text):
        text_bitmap = self.font.render(text, True, (0, 0, 0))
        screen.blit(text_bitmap, (self.x, self.y))
        self.y += self.line_height

    def reset(self):
        self.x = 10
        self.y = 10
        self.line_height = 15

text_print = TextPrint()

# ==========================
# Joystick Setup
# ==========================
pygame.joystick.init()
joysticks = {}

for i in range(pygame.joystick.get_count()):
    joy = pygame.joystick.Joystick(i)
    joy.init()
    joysticks[joy.get_instance_id()] = joy
    print("Joystick connected:", joy.get_name())

# ==========================
# Trim Slider Setup
# ==========================
ALIGN_OFFSET = DEFAULT_TRIM

SLIDER_X = 50
SLIDER_Y = image_height - 40
SLIDER_W = image_width - 200
SLIDER_H = 10
SLIDER_HANDLE_W = 20

dragging_slider = False

RESET_BTN_X = SLIDER_X + SLIDER_W + 30
RESET_BTN_Y = SLIDER_Y - 10
RESET_BTN_W = 80
RESET_BTN_H = 30

# ==========================
# HEADING COMMAND TEXT BOX
# ==========================
input_active = False
input_text = ""
input_box = pygame.Rect(50, image_height - 80, 200, 30)
font = pygame.font.Font(None, 28)

# ==========================
# Main Loop
# ==========================
done = False

while not done:

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True

        # ==========================
        # TEXT BOX EVENTS
        # ==========================
        if event.type == pygame.MOUSEBUTTONDOWN:
            if input_box.collidepoint(event.pos):
                input_active = True
            else:
                input_active = False

            mx, my = event.pos

            # Slider drag
            if SLIDER_Y - 10 <= my <= SLIDER_Y + 20:
                dragging_slider = True

            # Reset trim
            if RESET_BTN_X <= mx <= RESET_BTN_X + RESET_BTN_W and RESET_BTN_Y <= my <= RESET_BTN_Y + RESET_BTN_H:
                ALIGN_OFFSET = DEFAULT_TRIM

        if event.type == pygame.MOUSEBUTTONUP:
            dragging_slider = False

        # ==========================
        # TEXT INPUT
        # ==========================
        if event.type == pygame.KEYDOWN and input_active:
            if event.key == pygame.K_RETURN:
                try:
                    deg, sec = input_text.split()
                    deg = float(deg)
                    duration_ms = int(float(sec) * 1000)
                    send_heading(deg, duration_ms)
                except:
                    pass
                input_text = ""
            elif event.key == pygame.K_BACKSPACE:
                input_text = input_text[:-1]
            else:
                input_text += event.unicode

        # Joystick connect/disconnect
        if event.type == pygame.JOYDEVICEADDED:
            joy = pygame.joystick.Joystick(event.device_index)
            joy.init()
            joysticks[joy.get_instance_id()] = joy

        if event.type == pygame.JOYDEVICEREMOVED:
            if event.instance_id in joysticks:
                del joysticks[event.instance_id]

    # ==========================
    # SLIDER UPDATE
    # ==========================
    if dragging_slider:
        mx, my = pygame.mouse.get_pos()
        pos = max(SLIDER_X, min(mx, SLIDER_X + SLIDER_W))
        ALIGN_OFFSET = ((pos - SLIDER_X) / SLIDER_W) * 2 - 1
        ALIGN_OFFSET = round(ALIGN_OFFSET, 2)

    # ==========================
    # DRAW UI
    # ==========================
    screen.fill((255, 255, 255))
    screen.blit(background, (0, 0))
    text_print.reset()

    # ==========================
    # DEFAULT MOTOR VALUES
    # ==========================
    L_speed = 0
    L_dir   = 0
    R_speed = 0
    R_dir   = 0

    # ==========================
    # NES INPUTS
    # ==========================
    for joystick in joysticks.values():

        axis_0_val = joystick.get_axis(0)
        axis_4_val = joystick.get_axis(4)
        threshold = 0.9

        UP    = (axis_4_val <= -threshold)
        DOWN  = (axis_4_val >= threshold)
        LEFT  = (axis_0_val <= -threshold)
        RIGHT = (axis_0_val >= threshold)

        if UP:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, 1

        elif DOWN:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, -1

        elif LEFT:
            L_speed, L_dir = 255, -1
            R_speed, R_dir = 255, 1

        elif RIGHT:
            L_speed, L_dir = 255, 1
            R_speed, R_dir = 255, -1

        else:
            L_speed, L_dir = 0, 0
            R_speed, R_dir = 0, 0

        # Turning while moving
        if UP or DOWN:
            if RIGHT:
                R_speed = int(R_speed * 0.5)
            if LEFT:
                L_speed = int(L_speed * 0.5)

        # Trim
        trim_pwm = int(MAX_TRIM * abs(ALIGN_OFFSET))

        if ALIGN_OFFSET > 0:
            R_speed = max(0, R_speed - trim_pwm)
        elif ALIGN_OFFSET < 0:
            L_speed = max(0, L_speed - trim_pwm)

        R_speed = max(0, min(255, R_speed))
        L_speed = max(0, min(255, L_speed))

    # ==========================
    # SEND UDP PACKET
    # ==========================
    send_udp(R_speed, R_dir, L_speed, L_dir)

    # ==========================
    # DRAW TRIM SLIDER
    # ==========================
    pygame.draw.rect(screen, (50, 50, 50), (SLIDER_X, SLIDER_Y, SLIDER_W, SLIDER_H))

    handle_x = SLIDER_X + int((ALIGN_OFFSET + 1) / 2 * SLIDER_W)
    pygame.draw.rect(screen, (200, 0, 0),
                     (handle_x - SLIDER_HANDLE_W//2, SLIDER_Y - 5,
                      SLIDER_HANDLE_W, SLIDER_H + 10))

    text_print.tprint(screen, f"Trim: {ALIGN_OFFSET:+.2f}")

    # ==========================
    # RESET BUTTON
    # ==========================
    pygame.draw.rect(screen, (180, 180, 180),
                     (RESET_BTN_X, RESET_BTN_Y, RESET_BTN_W, RESET_BTN_H))
    pygame.draw.rect(screen, (0, 0, 0),
                     (RESET_BTN_X, RESET_BTN_Y, RESET_BTN_W, RESET_BTN_H), 2)

    reset_text = pygame.font.Font(None, 24).render("RESET", True, (0, 0, 0))
    screen.blit(reset_text, (RESET_BTN_X + 10, RESET_BTN_Y + 5))

    # ==========================
    # HEADING COMMAND TEXT BOX
    # ==========================
    pygame.draw.rect(screen, (255, 255, 255), input_box)
    pygame.draw.rect(screen, (0, 0, 0), input_box, 2)

    txt_surface = font.render(input_text, True, (0, 0, 0))
    screen.blit(txt_surface, (input_box.x + 5, input_box.y + 5))

    text_print.tprint(screen, "Enter:  <heading> <seconds>")

    pygame.display.flip()
    clock.tick(30)

stop_all()
pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Joystick connected: USB Gamepad


KeyboardInterrupt: 

: 

In [ ]:
import pygame
import socket
import requests
import time

# ==========================
# CONFIG
# ==========================
ROVER_IP = "192.168.4.1"
ROVER_PORT = 4210

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# ==========================
# NETWORK COMMANDS
# ==========================
def send_heading(deg, duration_ms):
    msg = f"HEAD,{deg},{duration_ms}"
    sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))

def send_stop():
    msg = "0,0,0,0"
    sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))

def get_status():
    try:
        r = requests.get(f"http://{ROVER_IP}/status", timeout=0.2)
        return r.json()
    except:
        return None

# ==========================
# PYGAME SETUP
# ==========================
pygame.init()
pygame.font.init()

WIDTH, HEIGHT = 600, 400
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Rover Controller")

font = pygame.font.Font(None, 28)
bigfont = pygame.font.Font(None, 32)

clock = pygame.time.Clock()

# ==========================
# TEXT INPUT BOX
# ==========================
input_box = pygame.Rect(20, HEIGHT - 60, 400, 40)
input_text = ""
input_active = False

# Buttons
send_button = pygame.Rect(430, HEIGHT - 60, 70, 40)
stop_button = pygame.Rect(510, HEIGHT - 60, 70, 40)

# Telemetry update timer
last_status_time = 0
status_cache = None

# ==========================
# MAIN LOOP
# ==========================
running = True
while running:
    screen.fill((240, 240, 240))

    # ==========================
    # TELEMETRY UPDATE
    # ==========================
    if time.time() - last_status_time > 0.5:
        status_cache = get_status()
        last_status_time = time.time()

    # ==========================
    # DRAW TELEMETRY
    # ==========================
    y = 20
    if status_cache:
        screen.blit(bigfont.render("Telemetry:", True, (0, 0, 0)), (20, y))
        y += 35

        screen.blit(font.render(f"Heading: {status_cache['heading']:.2f}°", True, (0, 0, 0)), (20, y))
        y += 25

        if status_cache["fix"]:
            gps_text = f"GPS: {status_cache['lat']:.6f}, {status_cache['lon']:.6f}"
        else:
            gps_text = "GPS: NO FIX"
        screen.blit(font.render(gps_text, True, (0, 0, 0)), (20, y))
        y += 25

        screen.blit(font.render(
            f"Motors: A={status_cache['A_speed']} dir={status_cache['A_dir']} | "
            f"B={status_cache['B_speed']} dir={status_cache['B_dir']}",
            True, (0, 0, 0)), (20, y))
        y += 25

        screen.blit(font.render(f"RSSI: {status_cache['RSSI']} dBm", True, (0, 0, 0)), (20, y))

    else:
        screen.blit(font.render("No telemetry...", True, (150, 0, 0)), (20, 20))

    # ==========================
    # DRAW INPUT BOX
    # ==========================
    pygame.draw.rect(screen, (255, 255, 255), input_box)
    pygame.draw.rect(screen, (0, 0, 0), input_box, 2)

    txt_surface = font.render(input_text, True, (0, 0, 0))
    screen.blit(txt_surface, (input_box.x + 5, input_box.y + 8))

    # ==========================
    # DRAW BUTTONS
    # ==========================
    pygame.draw.rect(screen, (200, 200, 200), send_button)
    pygame.draw.rect(screen, (0, 0, 0), send_button, 2)
    screen.blit(font.render("SEND", True, (0, 0, 0)),
                (send_button.x + 10, send_button.y + 10))

    pygame.draw.rect(screen, (255, 180, 180), stop_button)
    pygame.draw.rect(screen, (0, 0, 0), stop_button, 2)
    screen.blit(font.render("STOP", True, (0, 0, 0)),
                (stop_button.x + 10, stop_button.y + 10))

    # ==========================
    # EVENT HANDLING
    # ==========================
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        # Mouse click
        if event.type == pygame.MOUSEBUTTONDOWN:
            if input_box.collidepoint(event.pos):
                input_active = True
            else:
                input_active = False

            if send_button.collidepoint(event.pos):
                try:
                    deg, sec = input_text.split()
                    deg = float(deg)
                    duration_ms = int(float(sec) * 1000)
                    send_heading(deg, duration_ms)
                except:
                    print("Invalid command")
                input_text = ""

            if stop_button.collidepoint(event.pos):
                send_stop()

        # Keyboard input
        if event.type == pygame.KEYDOWN and input_active:
            if event.key == pygame.K_RETURN:
                try:
                    deg, sec = input_text.split()
                    deg = float(deg)
                    duration_ms = int(float(sec) * 1000)
                    send_heading(deg, duration_ms)
                except:
                    print("Invalid command")
                input_text = ""

            elif event.key == pygame.K_BACKSPACE:
                input_text = input_text[:-1]
            else:
                input_text += event.unicode

    pygame.display.flip()
    clock.tick(30)

pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
import pygame
import socket
import requests
import time
import math

# ==========================
# CONFIG
# ==========================
ROVER_IP = "192.168.4.1"
ROVER_PORT = 4210

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

def send_heading(deg, duration_ms):
    msg = f"HEAD,{deg},{duration_ms}"
    sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))

def send_stop():
    msg = "0,0,0,0"
    sock.sendto(msg.encode(), (ROVER_IP, ROVER_PORT))

def get_status():
    try:
        r = requests.get(f"http://{ROVER_IP}/status", timeout=0.2)
        return r.json()
    except:
        return None

# ==========================
# PYGAME SETUP
# ==========================
pygame.init()
pygame.font.init()

WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Rover Control Station")

font = pygame.font.Font(None, 24)
bigfont = pygame.font.Font(None, 32)

clock = pygame.time.Clock()

# ==========================
# UI ELEMENTS
# ==========================
# Telemetry panel
TELEM_X, TELEM_Y = 20, 20
TELEM_W, TELEM_H = 400, 400
telemetry_rect = pygame.Rect(TELEM_X, TELEM_Y, TELEM_W, TELEM_H)
telemetry_lines = []
telemetry_scroll = 0  # 0 = bottom (latest)

# Compass
COMPASS_CX = 650
COMPASS_CY = 250
COMPASS_R = 150

# Input fields
heading_box = pygame.Rect(20, HEIGHT - 80, 150, 35)
duration_box = pygame.Rect(190, HEIGHT - 80, 150, 35)
heading_text = ""
duration_text = ""

active_heading = False
active_duration = False

# Buttons
send_button = pygame.Rect(360, HEIGHT - 80, 80, 35)
stop_button = pygame.Rect(460, HEIGHT - 80, 80, 35)

# Command heading
command_heading = 0.0

# Actual heading smoothing
display_heading_actual = 0.0
heading_smooth_alpha = 0.15  # 0–1, higher = faster

# Compass drag
compass_dragging = False

# Telemetry polling
last_status_time = 0
status_cache = None

# ==========================
# HELPER FUNCTIONS
# ==========================
def angle_wrap_deg(a):
    while a < 0:
        a += 360
    while a >= 360:
        a -= 360
    return a

def smooth_heading(prev, new, alpha):
    # handle wrap-around
    diff = (new - prev + 540) % 360 - 180
    return angle_wrap_deg(prev + alpha * diff)

def add_telemetry_line(text):
    telemetry_lines.append(text)
    # keep last N lines
    if len(telemetry_lines) > 200:
        telemetry_lines.pop(0)

def draw_telemetry():
    pygame.draw.rect(screen, (230, 230, 230), telemetry_rect)
    pygame.draw.rect(screen, (0, 0, 0), telemetry_rect, 2)

    # visible lines
    line_height = 18
    max_lines = TELEM_H // line_height

    start_index = max(0, len(telemetry_lines) - max_lines - telemetry_scroll)
    end_index = max(0, len(telemetry_lines) - telemetry_scroll)

    y = TELEM_Y + 5
    for line in telemetry_lines[start_index:end_index]:
        surf = font.render(line, True, (0, 0, 0))
        screen.blit(surf, (TELEM_X + 5, y))
        y += line_height

def draw_compass(heading_actual, heading_cmd):
    # background
    pygame.draw.circle(screen, (245, 245, 245), (COMPASS_CX, COMPASS_CY), COMPASS_R)
    pygame.draw.circle(screen, (0, 0, 0), (COMPASS_CX, COMPASS_CY), COMPASS_R, 2)

    # tick marks every 30°
    for deg in range(0, 360, 30):
        rad = math.radians(deg - 90)
        x1 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 10)
        y1 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 10)
        x2 = COMPASS_CX + math.cos(rad) * (COMPASS_R - 2)
        y2 = COMPASS_CY + math.sin(rad) * (COMPASS_R - 2)
        pygame.draw.line(screen, (0, 0, 0), (x1, y1), (x2, y2), 2)

    # cardinal labels
    for label, deg in [("N", 0), ("E", 90), ("S", 180), ("W", 270)]:
        rad = math.radians(deg - 90)
        x = COMPASS_CX + math.cos(rad) * (COMPASS_R - 25)
        y = COMPASS_CY + math.sin(rad) * (COMPASS_R - 25)
        surf = bigfont.render(label, True, (0, 0, 0))
        rect = surf.get_rect(center=(x, y))
        screen.blit(surf, rect)

    # actual heading pointer (red)
    rad_act = math.radians(heading_actual - 90)
    x_act = COMPASS_CX + math.cos(rad_act) * (COMPASS_R - 35)
    y_act = COMPASS_CY + math.sin(rad_act) * (COMPASS_R - 35)
    pygame.draw.line(screen, (200, 0, 0), (COMPASS_CX, COMPASS_CY), (x_act, y_act), 4)

    # command heading pointer (blue)
    rad_cmd = math.radians(heading_cmd - 90)
    x_cmd = COMPASS_CX + math.cos(rad_cmd) * (COMPASS_R - 60)
    y_cmd = COMPASS_CY + math.sin(rad_cmd) * (COMPASS_R - 60)
    pygame.draw.line(screen, (0, 0, 200), (COMPASS_CX, COMPASS_CY), (x_cmd, y_cmd), 4)

    # labels
    label = font.render(f"Actual: {heading_actual:6.2f}°", True, (0, 0, 0))
    screen.blit(label, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 10))

    label2 = font.render(f"Command: {heading_cmd:6.2f}°", True, (0, 0, 0))
    screen.blit(label2, (COMPASS_CX - 80, COMPASS_CY + COMPASS_R + 30))

def compass_angle_from_mouse(mx, my):
    dx = mx - COMPASS_CX
    dy = my - COMPASS_CY
    angle = math.degrees(math.atan2(dy, dx)) + 90  # 0 at north
    return angle_wrap_deg(angle)

# ==========================
# MAIN LOOP
# ==========================
running = True
while running:
    screen.fill((240, 240, 240))

    # TELEMETRY UPDATE
    if time.time() - last_status_time > 0.3:
        status_cache = get_status()
        last_status_time = time.time()
        if status_cache:
            line = (f"Hdg {status_cache['heading']:.2f}° | "
                    f"GPS {'FIX' if status_cache['fix'] else 'NOFIX'} "
                    f"{status_cache['lat']:.6f},{status_cache['lon']:.6f} | "
                    f"A={status_cache['A_speed']}({status_cache['A_dir']}) "
                    f"B={status_cache['B_speed']}({status_cache['B_dir']}) | "
                    f"RSSI {status_cache['RSSI']} dBm")
            add_telemetry_line(line)

            # smooth actual heading
            display_heading_actual = smooth_heading(
                display_heading_actual,
                status_cache['heading'],
                heading_smooth_alpha
            )

    # DRAW TELEMETRY
    draw_telemetry()

    # DRAW COMPASS
    draw_compass(display_heading_actual, command_heading)

    # INPUT LABELS
    heading_label = font.render("Heading (deg):", True, (0, 0, 0))
    duration_label = font.render("Duration (sec):", True, (0, 0, 0))
    screen.blit(heading_label, (heading_box.x, heading_box.y - 20))
    screen.blit(duration_label, (duration_box.x, duration_box.y - 20))

    # INPUT BOXES
    pygame.draw.rect(screen, (255, 255, 255), heading_box)
    pygame.draw.rect(screen, (0, 0, 0), heading_box, 2)
    pygame.draw.rect(screen, (255, 255, 255), duration_box)
    pygame.draw.rect(screen, (0, 0, 0), duration_box, 2)

    heading_surf = font.render(heading_text, True, (0, 0, 0))
    duration_surf = font.render(duration_text, True, (0, 0, 0))
    screen.blit(heading_surf, (heading_box.x + 5, heading_box.y + 8))
    screen.blit(duration_surf, (duration_box.x + 5, duration_box.y + 8))

    # BUTTONS
    pygame.draw.rect(screen, (200, 200, 200), send_button)
    pygame.draw.rect(screen, (0, 0, 0), send_button, 2)
    screen.blit(font.render("SEND", True, (0, 0, 0)),
                (send_button.x + 15, send_button.y + 8))

    pygame.draw.rect(screen, (255, 180, 180), stop_button)
    pygame.draw.rect(screen, (0, 0, 0), stop_button, 2)
    screen.blit(font.render("STOP", True, (0, 0, 0)),
                (stop_button.x + 15, stop_button.y + 8))

    # EVENTS
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        # Mouse wheel for telemetry scroll
        if event.type == pygame.MOUSEWHEEL:
            telemetry_scroll -= event.y
            telemetry_scroll = max(0, min(telemetry_scroll, max(0, len(telemetry_lines) - 1)))

        if event.type == pygame.MOUSEBUTTONDOWN:
            mx, my = event.pos

            # input focus
            if heading_box.collidepoint(event.pos):
                active_heading = True
                active_duration = False
            elif duration_box.collidepoint(event.pos):
                active_heading = False
                active_duration = True
            else:
                active_heading = active_duration = False

            # buttons
            if send_button.collidepoint(event.pos):
                try:
                    deg = float(heading_text)
                    sec = float(duration_text)
                    send_heading(deg, int(sec * 1000))
                    add_telemetry_line(f"[CMD] HEAD {deg} for {sec}s")
                except:
                    add_telemetry_line("[ERR] Invalid heading/duration")

            if stop_button.collidepoint(event.pos):
                send_stop()
                add_telemetry_line("[CMD] STOP")

            # compass click
            dist = math.hypot(mx - COMPASS_CX, my - COMPASS_CY)
            if dist <= COMPASS_R:
                compass_dragging = True
                command_heading = compass_angle_from_mouse(mx, my)
                heading_text = f"{command_heading:.1f}"

        if event.type == pygame.MOUSEBUTTONUP:
            compass_dragging = False

        if event.type == pygame.MOUSEMOTION and compass_dragging:
            mx, my = event.pos
            command_heading = compass_angle_from_mouse(mx, my)
            heading_text = f"{command_heading:.1f}"

        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                running = False

            if active_heading:
                if event.key == pygame.K_BACKSPACE:
                    heading_text = heading_text[:-1]
                elif event.key == pygame.K_RETURN:
                    # do nothing special here
                    pass
                else:
                    heading_text += event.unicode

            elif active_duration:
                if event.key == pygame.K_BACKSPACE:
                    duration_text = duration_text[:-1]
                elif event.key == pygame.K_RETURN:
                    pass
                else:
                    duration_text += event.unicode

    pygame.display.flip()
    clock.tick(30)

pygame.quit()


C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
